    In this we introduce the modules used for the stiffness calculation.

In [1]:
using LinearAlgebra
using PyCall
using Statistics
using CSV
using DataFrames
using Plots
using DelimitedFiles
@pyimport numpy as np

In [2]:
# Importing personal modules
include("src/bdg_utilities.jl")
include("src/general_utils.jl")

Main.GeneralUtils

In [3]:
# Loading personal modules
using .BdGUtilities: generate_t_map, extract_geometry, build_H_twisted, generate_t_matrix
using .GeneralUtils: fermi_fn, show_all, show_matrix, check_rel_tol

using .BdGUtilities: create_bdg_ham_full, create_bdg_ham_full_twisted
using .BdGUtilities: sort_evecs_spinfull 

using .BdGUtilities: calc_delta_spinfull, calc_avg_n_spinfull, run_self_consistency_numpy_spinfull_twisted

f = fermi_fn

fermi_fn (generic function with 2 methods)

### Superfluid Stiffness ($D_s$)

The superfluid stiffness ($D_s$) measures the energy cost of twisting the superconducting phase. 
In 2D it controls phase coherence and determines the Berezinskii-Kosterlitz-Thouless (BKT) transition temperature.

A uniform phase twist ($\phi$) along the x-direction corresponds to a vector potential

$$
A_x = \frac{\phi}{L_x}.
$$

On a lattice this is implemented via the Peierls substitution:
$$
t_{ij} \rightarrow t_{ij}
\exp\left(i\phi\frac{x_j-x_i}{L_x}\right),
$$
where $(x_j-x_i)$ is the **physical bond displacement**,

Only bonds with nonzero $(x_j-x_i)$ contribute to $D_s^x$.

$$
\boxed{
D_s = 
\frac{L_x^2}{N}
\left.
\frac{\partial^2 F(\phi)}{\partial \phi^2}
\right|_{\phi=0}
}
$$

Here $F(\phi)$ is the free energy and (N) is the total number of lattice sites.

Numerically, the second derivative is evaluated using

$$
D_s = \frac{L_x^2}{N}
\frac{F(+\phi)+F(-\phi)-2F(0)}{\phi^2}
\qquad (\phi \ll 1).
$$

### Free energy in BdG theory

**Zero temperature**
$$
F(0) = -\frac{1}{2}\sum_n |E_n|
+
\sum_i \frac{|\Delta_i|^2}{U}
$$

**Finite temperature**
$$
F(T) = -T \sum_n \ln!\left(2\cosh\frac{E_n}{2T}\right) +
\sum_i \frac{|\Delta_i|^2}{U}
$$

### BKT transition criterion (2D)

The BKT transition temperature is determined by the universal jump condition:
$$
\boxed{
D_s(T_{\mathrm{BKT}}) = \frac{2}{\pi}T_{\mathrm{BKT}}
}
$$

In practice, $T_{\mathrm{BKT}}$ is obtained from the intersection of $D_s(T)$ with the line $2T/\pi$.

In [4]:
function bdg_spectrum_twist(
    deltaList, mu, nSites,
    nUp, nDn, U, impuritySite, J,
    x, y, neighbors, lambda, tMap;
    twist=0.0, K=0.0, Lx=nothing, isoMap=nothing, lambdaIso=0.0
)
    Hfull = create_bdg_ham_full_twisted(
        deltaList, mu, nSites,
        nUp, nDn, U, impuritySite, J,
        x, y, neighbors, lambda, tMap;
        twist=twist, K=K, Lx=Lx, isoMap=isoMap, lambdaIso=lambdaIso
    )

    evals, _ = np.linalg.eigh(Hfull)
    return evals
end


"""
    bdg_free_energy(evals, deltaList, U, T)

Compute the BdG free energy.

Arguments:
- evals: Complete BdG eigenvalues.
- deltaList: Converged order parameter at all sites.
- U: Attractive interaction strength.
- T: Temperature.

Returns:
- Free energy (ground-state energy if T == 0, finite-temperature free energy otherwise).
"""
function bdg_free_energy(evals, deltaList, U, T)
    if T == 0.0
        # T = 0 ground-state energy
        return -0.5 * sum(abs.(evals)) + sum(abs2.(deltaList)) / U
    else
        # finite-T free energy
        return -T * sum(log.(2 .* cosh.(abs.(evals) ./ (2T)))) +
               sum(abs2.(deltaList)) / U
    end
end


function free_energy_twist(
    twist, deltaList, mu, nSites,
    nUp, nDn, U, impuritySite, J,
    x, y, neighbors, lambda, tMap, T;
    K=0.0, Lx=nothing, isoMap=nothing, lambdaIso=0.0
)
    evals = bdg_spectrum_twist(
        deltaList, mu, nSites,
        nUp, nDn, U, impuritySite, J,
        x, y, neighbors, lambda, tMap;
        twist=twist, K=K, Lx=Lx, isoMap=isoMap, lambdaIso=lambdaIso
    )
    return bdg_free_energy(evals, deltaList, U, T)
end


function compute_Ds_helicity(
    deltaList, mu, nSites,
    nUp, nDn, U, impuritySite, J,
    x, y, neighbors, lambda, tMap, T;
    K=0.0, twist=1e-3, Lx=nothing, isoMap=nothing, lambdaIso=0.0
)
    xmin, xmax = minimum(x), maximum(x)
    Lx_val = isnothing(Lx) ? (xmax - xmin) : Lx

    F0 = free_energy_twist(0.0,  deltaList, mu, nSites, nUp, nDn, U, impuritySite, J,
                           x, y, neighbors, lambda, tMap, T;
                           K=K, Lx=Lx_val, isoMap=isoMap, lambdaIso=lambdaIso)

    Fp = free_energy_twist(+twist, deltaList, mu, nSites, nUp, nDn, U, impuritySite, J,
                           x, y, neighbors, lambda, tMap, T;
                           K=K, Lx=Lx_val, isoMap=isoMap, lambdaIso=lambdaIso)

    Fm = free_energy_twist(-twist, deltaList, mu, nSites, nUp, nDn, U, impuritySite, J,
                           x, y, neighbors, lambda, tMap, T;
                           K=K, Lx=Lx_val, isoMap=isoMap, lambdaIso=lambdaIso)

    Ds = (Lx_val^2 / nSites) * (Fp + Fm - 2F0) / twist^2
    return real(Ds)
end

compute_Ds_helicity (generic function with 1 method)

In [5]:
N             = 10
nSites        = 4 * N ^ 2

nUp          = ones(Float64, nSites)
nDn          = ones(Float64, nSites)

df = CSV.read("data/N$(N)/raw_df_square-octagon$N.csv", DataFrame)

x, y, neighbors = extract_geometry(df)
lambda = 0.9
lambdaIso = 0.0
t2 = 1.0

tMap = generate_t_map(neighbors, t2=t2)
twist = 1e-3

isoMap = nothing

if lambda == 0.0 && twist == 0.0
    deltaOld      = 0.7 * ones(Float64, nSites)
    isComplexCalc = false
else
    deltaOld      = 0.7 * ones(ComplexF64, nSites)
    isComplexCalc = true
end

impuritySite  = 3
mu             = -0.2
U             = 3

J             = 0.0
K             = 0.0
T             = 0.0

tMatFileName = "data/N$(N)/ham_$(N)_t2_$(t2)"
tMat = generate_t_matrix(tMatFileName); ## One doesn't need to use prebuilt hamiltonian. dataframe is enough to construct this matrix

In [6]:
deltaFinal, nUpFinal, nDnFinal, nAvg, evecs, evals, isConverged, endTime, count = 
    run_self_consistency_numpy_spinfull_twisted(
        deltaOld,
        mu, nSites,
        nUp, nDn,
        U, J, x, y,
        neighbors,
        lambda, lambdaIso,
        tMap, isoMap,
        impuritySite, T,
        twist=twist
    );

In [7]:
Ds = compute_Ds_helicity(deltaFinal, mu, nSites, nUp, nDn, U, 
    impuritySite, J, x, y, neighbors, lambda, tMap, T, twist=twist
    )

0.16943645469967636